# 01 - Schema Inspection & Ingestion

**Tujuan notebook ini:**

1. Scan folder `dataset/` dan kelompokkan 486 file CSV NBA/WNBA (play-by-play & shot detail, 1996-2024) berdasarkan *family* (nama dataset), musim (season), dan jenis game (regular/playoff).
2. Inspeksi skema tiap family lewat 1 file sampel: cek nama kolom, tipe data, dan jumlah baris.
3. Ingest penuh: baca semua file tiap family, tandai `season` & `game_type`, lalu union jadi satu DataFrame per family.
4. Sanity check: total baris & null count per kolom, serta deteksi apakah skema konsisten antar tahun dalam 1 family yang sama.
5. Simpan hasil ingestion sebagai Parquet (partitioned by `season`) ke `output/parquet/{family}/` untuk dipakai di notebook EDA/analisis selanjutnya.

Semua logic reusable ditaruh di `src/utils.py` (file discovery) dan `src/ingestion.py` (Spark pipeline), notebook ini hanya memanggil dan menampilkan hasilnya.

## Setup: SparkSession & environment

Catatan environment (Windows): PySpark butuh Java 8/11/17 (Java versi terbaru seperti 21+/23 tidak didukung Spark 3.5.x dan akan gagal start), serta `winutils.exe` + `hadoop.dll` untuk operasi filesystem lokal (termasuk menulis Parquet). Env var di bawah ini hanya diisi kalau belum di-set di sistem, supaya notebook tetap portable.

In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Java/Hadoop setup for local Windows runs (Spark 3.5.x needs Java 8/11/17, not 21+/23).
os.environ.setdefault("JAVA_HOME", r"C:\Program Files\Java\jre-1.8")
os.environ.setdefault("HADOOP_HOME", r"C:\hadoop")
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]

# Driver memory MUST be set before the JVM gateway launches (SparkSession.builder.config()
# is too late in local mode - the JVM is already up by then), so this goes through
# PYSPARK_SUBMIT_ARGS instead. Dataset is ~13GB of CSV, so give the single local-mode
# JVM (which doubles as driver+executor) plenty of headroom.
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 12g pyspark-shell"

DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")
PARQUET_DIR = os.path.join(OUTPUT_DIR, "parquet")
SCHEMA_SUMMARY_PATH = os.path.join(OUTPUT_DIR, "schema_summary.json")

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("NBA-WNBA-PlayByPlay-Ingestion")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

## Section 1: Dataset discovery

Scan `dataset/` dan kelompokkan file berdasarkan family (misal `nbastats`, `shotdetail`, `cdnnba`, dst), lalu tampilkan berapa file & rentang musim yang tersedia per family.

In [3]:
import pandas as pd

from src.utils import scan_dataset, summarize_families

families = scan_dataset(DATASET_DIR)
summary = summarize_families(families)

print(f"Jumlah family ditemukan: {len(families)}")
pd.DataFrame(summary).drop(columns=["seasons"])

Jumlah family ditemukan: 13


,family,total_files,regular_files,playoff_files,season_min,season_max
0,cdnnba,10,5,5,2020,2024
1,datanba,18,9,9,2016,2024
2,matchups,16,8,8,2017,2024
3,nbastats,58,29,29,1996,2024
4,nbastatsv3,58,29,29,1996,2024
5,pbpstats,50,25,25,2000,2024
6,shotdetail,58,29,29,1996,2024
7,wnba_cdnnba,6,3,3,2022,2024
8,wnba_datanba,16,8,8,2017,2024
9,wnba_nbastats,56,28,28,1997,2024


## Section 2: Schema inspection

Untuk tiap family, load 1 file sampel dengan PySpark lalu tampilkan `printSchema()`, `show(5)`, dan row count file sampel tersebut. Hasil ringkasan (kolom + tipe + jumlah baris) disimpan ke `output/schema_summary.json`.

In [4]:
from src.ingestion import inspect_schema, save_schema_summary

schema_summaries = []
for family, files in sorted(families.items()):
    schema_summaries.append(inspect_schema(spark, family, files))

save_schema_summary(schema_summaries, SCHEMA_SUMMARY_PATH)
print(f"\nSchema summary disimpan ke {SCHEMA_SUMMARY_PATH}")


=== Family: cdnnba (sample: cdnnba_po_2020.csv) ===
root
 |-- actionNumber: integer (nullable = true)
 |-- clock: string (nullable = true)
 |-- timeActual: timestamp (nullable = true)
 |-- period: integer (nullable = true)
 |-- periodType: string (nullable = true)
 |-- actionType: string (nullable = true)
 |-- subType: string (nullable = true)
 |-- qualifiers: string (nullable = true)
 |-- personId: integer (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- possession: integer (nullable = true)
 |-- scoreHome: integer (nullable = true)
 |-- scoreAway: integer (nullable = true)
 |-- edited: timestamp (nullable = true)
 |-- orderNumber: integer (nullable = true)
 |-- xLegacy: integer (nullable = true)
 |-- yLegacy: integer (nullable = true)
 |-- isFieldGoal: integer (nullable = true)
 |-- side: string (nullable = true)
 |-- description: string (nullable = true)
 |-- personIdsFilter: string (nullable = true)
 |-- teamId: integer (nullable = true)
 |-

+------------+-----------+---------------------+------+----------+----------+---------+----------------+--------+----------------+----------------+----------+---------+---------+-------------------+-----------+-------+-------+-----------+-----+--------------------------------------------------------------------+----------------------+----------+-----------+-------------------+---------------------+------------------------+----------+-----------+---------------------+-------------------+----------------------+--------------------+------------+----------+-----------+-----------------------+--------------+-----------+----------+-----------------+------------------+-------------------+-----------------+----------------+------------+---------------------+---------------------+-------------+---------------+-------------+---------------+-------------+-----+--------+
|actionNumber|clock      |timeActual           |period|periodType|actionType|subType  |qualifiers      |personId|x              

Row count (sample file only): 47443



=== Family: datanba (sample: datanba_po_2016.csv) ===
root
 |-- evt: integer (nullable = true)
 |-- cl: string (nullable = true)
 |-- de: string (nullable = true)
 |-- locX: integer (nullable = true)
 |-- locY: integer (nullable = true)
 |-- opt1: integer (nullable = true)
 |-- opt2: integer (nullable = true)
 |-- mtype: integer (nullable = true)
 |-- etype: integer (nullable = true)
 |-- opid: integer (nullable = true)
 |-- tid: integer (nullable = true)
 |-- pid: integer (nullable = true)
 |-- hs: integer (nullable = true)
 |-- vs: integer (nullable = true)
 |-- epid: integer (nullable = true)
 |-- oftid: integer (nullable = true)
 |-- PERIOD: integer (nullable = true)
 |-- GAME_ID: integer (nullable = true)

+---+-----+-------------------------------------------------------------------------+----+----+----+----+-----+-----+------+----------+------+---+---+------+----------+------+--------+
|evt|cl   |de                                                                       |locX|loc


=== Family: matchups (sample: matchups_po_2017.csv) ===
root
 |-- game_id: integer (nullable = true)
 |-- away_team_id: integer (nullable = true)
 |-- home_team_id: integer (nullable = true)
 |-- team_id: integer (nullable = true)
 |-- team_name: string (nullable = true)
 |-- team_city: string (nullable = true)
 |-- team_tricode: string (nullable = true)
 |-- team_slug: string (nullable = true)
 |-- person_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- family_name: string (nullable = true)
 |-- name_i: string (nullable = true)
 |-- player_slug: string (nullable = true)
 |-- position: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- jersey_num: integer (nullable = true)
 |-- matchups_person_id: integer (nullable = true)
 |-- matchups_first_name: string (nullable = true)
 |-- matchups_family_name: string (nullable = true)
 |-- matchups_name_i: string (nullable = true)
 |-- matchups_player_slug: string (nullable = true)
 |-- matchups_jers

Row count (sample file only): 14555


... (5 outputs left out)


=== Family: wnba_datanba (sample: wnba_datanba_po_2017.csv) ===
root
 |-- evt: integer (nullable = true)
 |-- cl: string (nullable = true)
 |-- de: string (nullable = true)
 |-- locX: integer (nullable = true)
 |-- locY: integer (nullable = true)
 |-- opt1: integer (nullable = true)
 |-- opt2: integer (nullable = true)
 |-- mtype: integer (nullable = true)
 |-- etype: integer (nullable = true)
 |-- opid: integer (nullable = true)
 |-- tid: integer (nullable = true)
 |-- pid: integer (nullable = true)
 |-- hs: integer (nullable = true)
 |-- vs: integer (nullable = true)
 |-- epid: integer (nullable = true)
 |-- oftid: integer (nullable = true)
 |-- PERIOD: integer (nullable = true)
 |-- GAME_ID: integer (nullable = true)

+---+-----+--------------------------------------------------------+----+----+----+----+-----+-----+------+----------+------+---+---+------+----------+------+----------+
|evt|cl   |de                                                      |locX|locY|opt1|opt2|mtype|etyp

Row count (sample file only): 1062

=== Family: wnba_nbastatsv3 (sample: wnba_nbastatsv3_po_1997.csv) ===
root
 |-- actionNumber: integer (nullable = true)
 |-- clock: string (nullable = true)
 |-- period: integer (nullable = true)
 |-- teamId: integer (nullable = true)
 |-- teamTricode: string (nullable = true)
 |-- personId: integer (nullable = true)
 |-- playerName: string (nullable = true)
 |-- playerNameI: string (nullable = true)
 |-- xLegacy: integer (nullable = true)
 |-- yLegacy: integer (nullable = true)
 |-- shotDistance: integer (nullable = true)
 |-- shotResult: string (nullable = true)
 |-- isFieldGoal: integer (nullable = true)
 |-- scoreHome: integer (nullable = true)
 |-- scoreAway: integer (nullable = true)
 |-- pointsTotal: integer (nullable = true)
 |-- location: string (nullable = true)
 |-- description: string (nullable = true)
 |-- actionType: string (nullable = true)
 |-- subType: string (nullable = true)
 |-- videoAvailable: integer (nullable = true)
 |-- actio


=== Family: wnba_pbpstats (sample: wnba_pbpstats_po_2009.csv) ===
root
 |-- ENDTIME: string (nullable = true)
 |-- EVENTS: string (nullable = true)
 |-- FG2A: integer (nullable = true)
 |-- FG2M: integer (nullable = true)
 |-- FG3A: integer (nullable = true)
 |-- FG3M: integer (nullable = true)
 |-- GAMEDATE: date (nullable = true)
 |-- GAMEID: integer (nullable = true)
 |-- NONSHOOTINGFOULSTHATRESULTEDINFTS: integer (nullable = true)
 |-- OFFENSIVEREBOUNDS: integer (nullable = true)
 |-- OPPONENT: string (nullable = true)
 |-- PERIOD: integer (nullable = true)
 |-- SHOOTINGFOULSDRAWN: integer (nullable = true)
 |-- STARTSCOREDIFFERENTIAL: integer (nullable = true)
 |-- STARTTIME: timestamp (nullable = true)
 |-- STARTTYPE: string (nullable = true)
 |-- TURNOVERS: integer (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- URL: string (nullable = true)

+--------------------------------------------------------------------------------------------------+-------------------

+-----------------+----------+-------------+---------+--------------+----------+---------------+------+-----------------+-----------------+-----------+-----------+--------------+---------------------+--------------+---------------+-------------+-----+-----+-------------------+--------------+---------+---+---+
|GRID_TYPE        |GAME_ID   |GAME_EVENT_ID|PLAYER_ID|PLAYER_NAME   |TEAM_ID   |TEAM_NAME      |PERIOD|MINUTES_REMAINING|SECONDS_REMAINING|EVENT_TYPE |ACTION_TYPE|SHOT_TYPE     |SHOT_ZONE_BASIC      |SHOT_ZONE_AREA|SHOT_ZONE_RANGE|SHOT_DISTANCE|LOC_X|LOC_Y|SHOT_ATTEMPTED_FLAG|SHOT_MADE_FLAG|GAME_DATE|HTM|VTM|
+-----------------+----------+-------------+---------+--------------+----------+---------------+------+-----------------+-----------------+-----------+-----------+--------------+---------------------+--------------+---------------+-------------+-----+-----+-------------------+--------------+---------+---+---+
|Shot Chart Detail|1029700004|2            |100058   |Vicky Bullett

## Section 3: Full ingestion

Baca SEMUA file tiap family (bukan cuma sampel), tambahkan kolom `season` & `game_type` dari nama file, lalu union jadi satu DataFrame per family. File dibaca satu per satu (bukan 1 wildcard glob) karena beberapa family (misal `cdnnba`, `datanba`) ternyata skemanya berubah antar tahun - `unionByName(allowMissingColumns=True)` dipakai supaya kolom tetap sejajar berdasarkan nama, bukan posisi.

In [5]:
from src.ingestion import ingest_all

dataframes, schema_reports = ingest_all(spark, families)

row_counts = []
for family, df in sorted(dataframes.items()):
    row_counts.append({"family": family, "union_row_count": df.count()})

pd.DataFrame(row_counts)

,family,union_row_count
0,cdnnba,3570026
1,datanba,5456646
2,matchups,1886689
3,nbastats,16699012
4,nbastatsv3,17612816
5,pbpstats,47665205
6,shotdetail,6094438
7,wnba_cdnnba,349117
8,wnba_datanba,698463
9,wnba_nbastats,2421013


In [6]:
# Konsistensi skema antar file dalam 1 family yang sama
pd.DataFrame(schema_reports)

,family,num_files,distinct_schema_variants,schema_consistent
0,cdnnba,10,3,False
1,datanba,18,3,False
2,matchups,16,1,True
3,nbastats,58,1,True
4,nbastatsv3,58,2,False
5,pbpstats,50,1,True
6,shotdetail,58,1,True
7,wnba_cdnnba,6,3,False
8,wnba_datanba,16,3,False
9,wnba_nbastats,56,1,True


## Section 4: Sanity check

Hitung total baris dan null count per kolom untuk tiap DataFrame hasil ingestion, supaya ketahuan kolom mana yang banyak kosong atau ada anomali.

In [7]:
from src.ingestion import sanity_check

sanity_results = {}
for family, df in sorted(dataframes.items()):
    result = sanity_check(family, df)
    sanity_results[family] = result
    print(f"\n=== {family} | total_rows={result['total_rows']} ===")
    null_df = pd.DataFrame(
        sorted(result["null_counts"].items(), key=lambda x: -x[1]),
        columns=["column", "null_count"],
    )
    null_df["null_pct"] = (null_df["null_count"] / result["total_rows"] * 100).round(2)
    display(null_df)


=== cdnnba | total_rows=3570026 ===


,column,null_count,null_pct
0,value,3566593,99.90
1,jumpBallRecoverdPersonId,3559204,99.70
2,jumpBallRecoveredName,3558735,99.68
3,jumpBallWonPlayerName,3558735,99.68
4,jumpBallWonPersonId,3558735,99.68
...,...,...,...
57,isFieldGoal,7,0.00
58,clock,5,0.00
59,actionNumber,0,0.00
60,season,0,0.00



=== datanba | total_rows=5456646 ===


,column,null_count,null_pct
0,opid,4694821,86.04
1,epid,4150829,76.07
2,wallclk,3580544,65.62
3,opt3,3580544,65.62
4,opt4,3580544,65.62
5,pts,3580544,65.62
6,ord,603364,11.06
7,evt,0,0.00
8,cl,0,0.00
9,de,0,0.00



=== matchups | total_rows=1886689 ===


,column,null_count,null_pct
0,comment,1886689,100.00
1,position,953017,50.51
2,jersey_num,1938,0.10
3,matchups_jersey_num,1924,0.10
4,game_id,0,0.00
5,away_team_id,0,0.00
6,home_team_id,0,0.00
7,team_id,0,0.00
8,team_name,0,0.00
9,team_city,0,0.00


... (16 outputs left out)


=== wnba_pbpstats | total_rows=4424557 ===


,column,null_count,null_pct
0,DESCRIPTION,4424557,100.00
1,URL,4424557,100.00
2,DESCRIPTION17,4424557,100.00
3,GAMEID7,4424537,100.00
4,PERIOD11,4424537,100.00
5,scoreHome,4424476,100.00
6,scoreAway,4424476,100.00
7,shotResult,4424417,100.00
8,subType,4424249,99.99
9,teamTricode,4424243,99.99



=== wnba_shotdetail | total_rows=814089 ===


,column,null_count,null_pct
0,GRID_TYPE,0,0.0
1,GAME_ID,0,0.0
2,GAME_EVENT_ID,0,0.0
3,PLAYER_ID,0,0.0
4,PLAYER_NAME,0,0.0
5,TEAM_ID,0,0.0
6,TEAM_NAME,0,0.0
7,PERIOD,0,0.0
8,MINUTES_REMAINING,0,0.0
9,SECONDS_REMAINING,0,0.0


## Section 5: Save to Parquet

Tulis tiap DataFrame family ke `output/parquet/{family}/`, dipartisi berdasarkan kolom `season`, supaya bisa dibaca efisien di notebook EDA selanjutnya tanpa parsing ulang CSV.

In [8]:
from src.ingestion import save_family_parquet

for family, df in sorted(dataframes.items()):
    path = save_family_parquet(df, family, PARQUET_DIR)
    print(f"Saved {family} -> {path}")

Saved cdnnba -> E:\KULIAH\SEM 7\BIG DATA\kelompokn1\output\parquet\cdnnba


Saved datanba -> E:\KULIAH\SEM 7\BIG DATA\kelompokn1\output\parquet\datanba


Saved matchups -> E:\KULIAH\SEM 7\BIG DATA\kelompokn1\output\parquet\matchups


Saved nbastats -> E:\KULIAH\SEM 7\BIG DATA\kelompokn1\output\parquet\nbastats


Saved nbastatsv3 -> E:\KULIAH\SEM 7\BIG DATA\kelompokn1\output\parquet\nbastatsv3


Saved pbpstats -> E:\KULIAH\SEM 7\BIG DATA\kelompokn1\output\parquet\pbpstats


... (3 outputs left out)

Saved wnba_nbastats -> E:\KULIAH\SEM 7\BIG DATA\kelompokn1\output\parquet\wnba_nbastats


Saved wnba_nbastatsv3 -> E:\KULIAH\SEM 7\BIG DATA\kelompokn1\output\parquet\wnba_nbastatsv3


Saved wnba_pbpstats -> E:\KULIAH\SEM 7\BIG DATA\kelompokn1\output\parquet\wnba_pbpstats


Saved wnba_shotdetail -> E:\KULIAH\SEM 7\BIG DATA\kelompokn1\output\parquet\wnba_shotdetail


In [9]:
spark.stop()